# Naive Bayes Classification in Python

Dataset from Kaggle : **"Netflix Movies and TV Shows"** by *Shivam Bansal*  
Source: https://www.kaggle.com/datasets/shivamb/netflix-shows (requires login)

This version is set up to **prevent data leakage** with a 60 / 20 / 20 Train / Validation / Test split -- information from the Validation and Test Sets never reaches the exploration, and the Test Set never reaches the model or any choice we make.

---

### Essential Libraries

Let us begin by importing the essential Python Libraries.

> NumPy : Library for Numeric Computations in Python  
> Pandas : Library for Data Acquisition and Preparation  
> Matplotlib : Low-level library for Data Visualization  
> Seaborn : Higher-level library for Data Visualization  

In [ ]:
# Basic Libraries
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt # we only need pyplot
sb.set() # set the default Seaborn style for graphics

In [ ]:
# Print library package version
print("numpy version = "+np.__version__)
print("pandas version = "+pd.__version__)
print("seaborn version = "+sb.__version__)

---

### Import the Dataset

The dataset is in CSV format; hence we use the `read_csv` function from Pandas.  
Immediately after importing, take a quick look at the data using the `head` function.

In [ ]:
netflixdata = pd.read_csv('netflix_titles.csv')
netflixdata.head()

Description of the dataset, as available on Kaggle, is as follows.

> **show_id** : Unique ID for each title  
> **type** : Whether the title is a Movie or a TV Show  
> **title** : Name of the title  
> **director** : Director(s) of the title  
> **cast** : Main actors in the title  
> **country** : Country or countries where the title was produced  
> **date_added** : Date the title was added to Netflix  
> **release_year** : Year the title was originally released  
> **rating** : Age rating of the title (e.g. TV-MA, PG-13)  
> **duration** : Minutes for movies, number of seasons for TV shows  
> **listed_in** : Genres of the title, separated by commas  
> **description** : Short summary of the title  

---

Check the vital statistics of the dataset using the `type` and `shape` attributes.

In [ ]:
print("Data type : ", type(netflixdata))
print("Data dims : ", netflixdata.shape)

Check the variables (and their types) in the dataset using the `dtypes` attribute.

In [ ]:
print(netflixdata.dtypes)

---

## Clean the Dataset

Check the dataset for missing values, duplicates and invalid entries.  

**Leakage rule :** cleaning here only looks at **one row at a time** (is this rating valid? is this description missing?) or removes **repeated rows**. Nothing is calculated from the dataset as a whole, so cleaning before the split does not leak anything.

In [ ]:
# Information about the Variables
netflixdata.info()

In [ ]:
# Missing values in each Variable
print(netflixdata.isnull().sum())

Visualize the missing values, to see at a glance which variables have gaps.

In [ ]:
# Missing values in each Variable, as a bar chart
missing = netflixdata.isnull().sum().sort_values()
f = plt.figure(figsize=(16, 8))
sb.barplot(x = missing.values, y = missing.index, orient = "h")
plt.xlabel("Number of missing values")

In [ ]:
# Ratings in the Dataset (including missing ones)
print(netflixdata["rating"].value_counts(dropna = False))

# Count plot of all Ratings -- invalid entries like "74 min", NR and UR stand out
sb.catplot(y = "rating", data = netflixdata, kind = "count",
           order = netflixdata["rating"].value_counts().index, height = 8)

#### Movies and TV Shows in the Dataset

In [ ]:
# Types in the Dataset
print(netflixdata["type"].value_counts())
sb.catplot(y = "type", data = netflixdata, kind = "count")

#### Remove Unusable Titles

Keep only titles with a valid age rating, a description and a genre.

In [ ]:
# Valid Age Ratings
valid_ratings = ["TV-Y", "TV-Y7", "TV-Y7-FV", "TV-G", "G", "PG", "TV-PG",
                 "PG-13", "TV-14", "R", "TV-MA", "NC-17"]

# Keep only titles with a valid rating, a description and a genre
netflixclean = netflixdata[netflixdata["rating"].isin(valid_ratings)]
netflixclean = netflixclean.dropna(subset = ["description", "listed_in"])

# Check the sample sizes
print("Before Cleaning :", netflixdata.shape)
print("After Cleaning  :", netflixclean.shape)

#### Remove Duplicate and Near-Duplicate Descriptions

**Why this prevents leakage :** if the same (or almost the same) description is in both the Train Set and the Test Set, the model is tested on text it has already seen, and the Test accuracy looks better than it really is.  

Exact copies are easy to find. Near-copies are the same title in another language or version (e.g. *Game Over (Hindi Version)* and *Game Over (Telugu Version)*), where only a word or two differs.  
We catch them by comparing the **first 12 words** of each description, in lower case with punctuation removed.

In [ ]:
# Remove exact duplicate descriptions
before = len(netflixclean)
netflixclean = netflixclean.drop_duplicates(subset = "description")
print("Exact duplicates removed :", before - len(netflixclean))

# Build a simple key : first 12 words, lower case, letters only
desc_key = netflixclean["description"].str.lower()
desc_key = desc_key.str.replace(r"[^a-z ]", "", regex = True)
desc_key = desc_key.str.split().str[:12].str.join(" ")

# Show a few near-duplicate pairs before removing them
near_dups = netflixclean[desc_key.duplicated(keep = False)]
print("Titles in near-duplicate groups :", len(near_dups))
near_dups.assign(key = desc_key).sort_values("key")[["title", "rating", "description"]].head(6)

In [ ]:
# Keep only the first title of each near-duplicate group
before = len(netflixclean)
netflixclean = netflixclean[desc_key.duplicated() == False]
print("Near-duplicates removed :", before - len(netflixclean))
print("Final Dataset :", netflixclean.shape)

---

## Set up the Classification Problem

Response Variable : **Adult** (whether the title is rated 18+)     
Predictor Features : **description** and **listed_in** (genre)    

A title is **18+** if its rating is *R*, *TV-MA* or *NC-17*, and **Not 18+** otherwise.

In [ ]:
# Create the Response Variable (row by row : uses only that title's own rating)
netflixclean["adult"] = netflixclean["rating"].isin(["R", "TV-MA", "NC-17"])
netflixclean["adult"] = netflixclean["adult"].map({True : "18+", False : "Not 18+"})

#### Combine Description and Genre

Each genre becomes a single token with a `g_` prefix -- e.g. *International Movies* becomes `g_International_Movies`.  
This is also row by row (a fixed text rule, nothing learnt from the data), so it does not leak.

In [ ]:
# Convert the Genres of each title into Genre tokens
def genre_tokens(genres):
    tokens = []
    for g in genres.split(", "):
        g = g.replace(" ", "_").replace("'", "").replace("-", "_").replace("&", "and")
        tokens.append("g_" + g)
    return " ".join(tokens)

netflixclean["genre"] = netflixclean["listed_in"].apply(genre_tokens)

# Combine Description and Genre into a single text
netflixclean["text"] = netflixclean["description"] + " " + netflixclean["genre"]

# Check the first title
print(netflixclean["listed_in"].iloc[0])
print(netflixclean["text"].iloc[0])

---

## Split into Train, Validation and Test (60 / 20 / 20) -- FIRST

**Why this prevents leakage :** we split **before** any exploration or modelling, into three sets with three separate jobs.  

> **Train (60%)** : explore the data and fit the model  
> **Validation (20%)** : compare settings (`alpha`, which Predictors) and pick the best  
> **Test (20%)** : locked away, used **once** at the very end for the final score  

`train_test_split` only makes two sets, so we split twice : first lock away the Test Set (20%), then split the remaining 80% into Train and Validation.  
`stratify` keeps the same proportion of 18+ and Not 18+ titles in every set; `random_state` makes the split repeatable.

In [ ]:
# Import train_test_split from sklearn
from sklearn.model_selection import train_test_split

# First split : lock away the Test Set (20% of all titles)
netflix_rest, netflix_test = train_test_split(netflixclean, test_size = 0.2,
                                              stratify = netflixclean["adult"], random_state = 42)

# Second split : the remaining 80% into Train (60% of all) and Validation (20% of all)
# 0.25 of the remaining 80% = 20% of the full dataset
netflix_train, netflix_val = train_test_split(netflix_rest, test_size = 0.25,
                                              stratify = netflix_rest["adult"], random_state = 42)

# Check the sample sizes
print("Train Set      :", netflix_train.shape)
print("Validation Set :", netflix_val.shape)
print("Test Set       :", netflix_test.shape)

In [ ]:
# Check that every set keeps the same class proportions
print("Train      :", netflix_train["adult"].value_counts(normalize = True).round(3).to_dict())
print("Validation :", netflix_val["adult"].value_counts(normalize = True).round(3).to_dict())
print("Test       :", netflix_test["adult"].value_counts(normalize = True).round(3).to_dict())

---

## Explore the Dataset -- Train Set only

**Why this prevents leakage :** every number and plot below is calculated on `netflix_train`.  
If we explored the whole dataset, patterns from the Validation or Test titles (e.g. which genres are mostly 18+) could steer our choices, and those sets would no longer be truly unseen.

#### Classes in the Train Set and the Baseline

In [ ]:
# Titles in each Class (Train Set)
print(netflix_train["adult"].value_counts())
sb.catplot(y = "adult", data = netflix_train, kind = "count")

In [ ]:
# Baseline : always guess the most common class in the Train Set
majority_class = netflix_train["adult"].value_counts().idxmax()
print("Majority class \t\t:", majority_class)
print("Baseline Accuracy \t:", netflix_train["adult"].value_counts(normalize = True).max())

#### Length of the Description

In [ ]:
# Number of words in each Description (Train Set)
desc_len = pd.DataFrame(netflix_train["description"].str.split().str.len())
desc_len.columns = ["Length"]

# Summary Statistics for Description Length
desc_len.describe()

In [ ]:
# Plot the distributions of Description Length
f, axes = plt.subplots(1, 3, figsize=(24, 6))
sb.boxplot(data = desc_len, orient = "h", ax = axes[0])
sb.histplot(data = desc_len, ax = axes[1])
sb.violinplot(data = desc_len, orient = "h", ax = axes[2])

In [ ]:
# Create a joint dataframe by concatenating Length and Adult
lenDF = pd.concat([desc_len, netflix_train["adult"]], axis = 1).reindex(desc_len.index)

# Joint Boxplot of Length against Adult
f = plt.figure(figsize=(18, 6))
sb.boxplot(x = "Length", y = "adult", data = lenDF, orient = "h")

#### Most Common Words in each Class

In [ ]:
# Import CountVectorizer from sklearn.feature_extraction.text
from sklearn.feature_extraction.text import CountVectorizer

# Top 15 words (ignoring common English words) in each Class -- Train Set only
for cls in ["18+", "Not 18+"]:
    words = CountVectorizer(stop_words = "english")
    counts = words.fit_transform(netflix_train[netflix_train["adult"] == cls]["description"])
    top = pd.Series(counts.sum(axis = 0).A1, index = words.get_feature_names_out())
    print(cls, ":", list(top.nlargest(15).index))
    print()

#### Genres against Adult

Share of 18+ titles within each genre (genres with at least 80 titles in the Train Set).

In [ ]:
# One row for each Genre of each title (Train Set)
genreDF = netflix_train[["listed_in", "adult"]].copy()
genreDF["listed_in"] = genreDF["listed_in"].str.split(", ")
genreDF = genreDF.explode("listed_in").reset_index(drop = True)

# Count the titles in each Genre and Class
genre_counts = genreDF.groupby(["listed_in", "adult"]).size().unstack(fill_value = 0)
genre_counts = genre_counts[genre_counts.sum(axis = 1) >= 80]

# Share of 18+ titles in each Genre
genre_share = genre_counts["18+"] / genre_counts.sum(axis = 1)
genre_share = genre_share.sort_values()
print(genre_share.round(2))

In [ ]:
# Plot the Share of 18+ titles in each Genre
f = plt.figure(figsize=(12, 12))
sb.barplot(x = genre_share.values, y = genre_share.index, orient = "h")
plt.axvline(0.5, color = "black", linestyle = "--")
plt.xlabel("Share of 18+ titles (Train Set)")

---

## Naive Bayes Classifier

#### Convert Text to Word Counts

A model can only work with numbers, so each title's text is converted into word counts using `CountVectorizer`.  

**Why this prevents leakage :** the vocabulary is learnt from the **Train Set only** (`fit_transform`).  
The Validation Set is only **transformed** with that vocabulary, so a word that appears only in Validation titles teaches the model nothing.

In [ ]:
# Import CountVectorizer, the model and the scoring function from sklearn
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

# Learn the vocabulary from the Train Set only
vectorizer = CountVectorizer(token_pattern = r"(?u)\b\w+\b")
X_train_counts = vectorizer.fit_transform(netflix_train["text"])   # fit on Train
X_val_counts = vectorizer.transform(netflix_val["text"])           # transform Validation only

# Check the sizes (titles, words in vocabulary)
print("Train Counts      :", X_train_counts.shape)
print("Validation Counts :", X_val_counts.shape)

#### Choose the Smoothing Parameter -- on the Validation Set

`alpha` adds a small count to every word so no word has a probability of zero.  
Train a model for each `alpha` on the Train Set, score it on the Validation Set, and keep the best.  

**Why this prevents leakage :** `alpha` is chosen with the **Validation Set**, never the Test Set. Picking settings by Test scores would tune the model to the Test Set.

In [ ]:
# Try different values of alpha : train on Train, score on Validation
alpha_scores = {}
for alpha in [0.1, 0.5, 1, 2, 3]:
    nbmodel = MultinomialNB(alpha = alpha)
    nbmodel.fit(X_train_counts, netflix_train["adult"])
    val_pred = nbmodel.predict(X_val_counts)
    alpha_scores[alpha] = f1_score(netflix_val["adult"], val_pred, average = "macro")
    print("alpha =", alpha, "\tValidation Macro-F1 :", round(alpha_scores[alpha], 3))

# Keep the best alpha
best_alpha = max(alpha_scores, key = alpha_scores.get)
print()
print("Best alpha :", best_alpha)

#### Does Genre Help? -- decided on the Validation Set

Compare three Predictors -- **description only**, **genre only**, and **description + genre** -- trained on the Train Set and scored on the Validation Set.  
Each Predictor gets its own vocabulary, again learnt from the Train Set only.

In [ ]:
# Compare Predictors : train on Train, score on Validation
for var in ["description", "genre", "text"]:
    vec = CountVectorizer(token_pattern = r"(?u)\b\w+\b")
    train_counts = vec.fit_transform(netflix_train[var])   # fit on Train
    val_counts = vec.transform(netflix_val[var])           # transform Validation only

    model = MultinomialNB(alpha = best_alpha)
    model.fit(train_counts, netflix_train["adult"])
    val_pred = model.predict(val_counts)

    print("Predictor :", var)
    print("Validation Accuracy \t:", round(model.score(val_counts, netflix_val["adult"]), 3))
    print("Validation Macro-F1 \t:", round(f1_score(netflix_val["adult"], val_pred, average = "macro"), 3))
    print()

#### Train the Final Model

All choices are now made (`alpha`, and description + genre as the Predictor), so the Validation Set has done its job.  
We retrain on **Train + Validation (80%)** so the final model learns from as much data as possible -- the Test Set is still untouched.  

The word counter is re-fitted on Train + Validation, and the Test Set is only **transformed** with that vocabulary.

In [ ]:
# Combine Train and Validation into the final training data
netflix_final = pd.concat([netflix_train, netflix_val])

# Extract Response and Predictor for Final Train and Test
X_final, y_final = netflix_final["text"], netflix_final["adult"]
X_test, y_test = netflix_test["text"], netflix_test["adult"]

# Learn the vocabulary from Train + Validation only
vectorizer = CountVectorizer(token_pattern = r"(?u)\b\w+\b")
X_final_counts = vectorizer.fit_transform(X_final)   # fit on Train + Validation
X_test_counts = vectorizer.transform(X_test)         # transform Test only

# Check the sizes (titles, words in vocabulary)
print("Final Train Counts :", X_final_counts.shape)
print("Test Counts        :", X_test_counts.shape)

In [ ]:
# Create and Train the final Naive Bayes model with the chosen alpha
nbmodel = MultinomialNB(alpha = best_alpha)
nbmodel.fit(X_final_counts, y_final)

#### Goodness of Fit of the Model

Check how good the predictions are on the data the final model was trained on (Train + Validation).    
Metrics : Classification Accuracy and Confusion Matrix.

In [ ]:
# Import confusion_matrix from sklearn
from sklearn.metrics import confusion_matrix

# Predict Adult corresponding to Text (Train + Validation)
y_final_pred = nbmodel.predict(X_final_counts)

# Print the Classification Accuracy
print("Classification Accuracy \t:", nbmodel.score(X_final_counts, y_final))

# Plot the two-way Confusion Matrix
f = plt.figure(figsize=(8, 6))
sb.heatmap(confusion_matrix(y_final, y_final_pred, labels = ["18+", "Not 18+"]),
           annot = True, fmt=".0f", annot_kws={"size": 18},
           xticklabels = ["18+", "Not 18+"], yticklabels = ["18+", "Not 18+"])
plt.title("Train + Validation Set")
plt.xlabel("Predicted")
plt.ylabel("Actual")

Check how good the predictions are on the Test Set -- the **first and only time** the Test Set is used.    
Metrics : Classification Accuracy and Confusion Matrix.

In [ ]:
# Predict Adult corresponding to Text Test
y_test_pred = nbmodel.predict(X_test_counts)

# Print the Classification Accuracy
print("Classification Accuracy \t:", nbmodel.score(X_test_counts, y_test))

# Plot the two-way Confusion Matrix
f = plt.figure(figsize=(8, 6))
sb.heatmap(confusion_matrix(y_test, y_test_pred, labels = ["18+", "Not 18+"]),
           annot = True, fmt=".0f", annot_kws={"size": 18},
           xticklabels = ["18+", "Not 18+"], yticklabels = ["18+", "Not 18+"])
plt.title("Test Set")
plt.xlabel("Predicted")
plt.ylabel("Actual")

Look at Precision, Recall and F1-score for each class on the Test Set.

In [ ]:
# Import classification_report from sklearn
from sklearn.metrics import classification_report

print(classification_report(y_test, y_test_pred))

---

## What the Model Learnt

Compare the probability of each word in the two classes (learnt from Train + Validation).  
Words with a much higher probability in one class push the prediction towards that class.

In [ ]:
# Words in the vocabulary
words = vectorizer.get_feature_names_out()

# Difference in log-probability : positive means more typical of 18+
adult_idx = list(nbmodel.classes_).index("18+")
other_idx = 1 - adult_idx
diff = pd.Series(nbmodel.feature_log_prob_[adult_idx] - nbmodel.feature_log_prob_[other_idx], index = words)

# Split into Genre tokens and Description words
genre_diff = diff[diff.index.str.startswith("g_")]
word_diff = diff[~diff.index.str.startswith("g_")]

print("Genres pointing to 18+ \t\t:", list(genre_diff.nlargest(10).index))
print("Genres pointing to Not 18+ \t:", list(genre_diff.nsmallest(10).index))
print()
print("Words pointing to 18+ \t\t:", list(word_diff.nlargest(10).index))
print("Words pointing to Not 18+ \t:", list(word_diff.nsmallest(10).index))

---

## Error Analysis

Which of the original ratings are misclassified most often on the Test Set?  
This only **describes** the final result -- we do not go back and change the model because of it, so it does not leak.

In [ ]:
# Summarize the Actuals and Predictions
y_test_pred = pd.DataFrame(y_test_pred, columns = ["PredAdult"], index = y_test.index)
netflix_acc = pd.concat([netflix_test[["title", "rating", "listed_in", "description", "adult"]],
                         y_test_pred], axis = 1)
netflix_acc["Correct"] = netflix_acc["adult"] == netflix_acc["PredAdult"]

# Accuracy for each original Rating
print(netflix_acc.groupby("rating")["Correct"].mean().sort_values().round(2))

In [ ]:
# A few Misclassified Titles
netflix_acc[netflix_acc["Correct"] == False].head(10)

---

## Prediction for Specific Titles

Let's predict the class for a few specific titles, along with the Class Probabilities.  

**Why this prevents leakage :** the titles are picked from the **Test Set**. A title from the Train or Validation Set would show how well the model remembers, not how well it predicts.

In [ ]:
# Extract Titles for Prediction (all four are in the Test Set)
netflix_pred = netflix_test[netflix_test["title"].isin(["Black Mirror", "Squid Game", "Never Have I Ever", "Octonauts: Above & Beyond"])]

# Predict Response corresponding to Predictors
X_pred = vectorizer.transform(netflix_pred["text"])
y_pred = nbmodel.predict(X_pred)

# Summarize the Actuals and Predictions
y_pred = pd.DataFrame(y_pred, columns = ["PredAdult"], index = netflix_pred.index)
netflix_pred_acc = pd.concat([netflix_pred[["title", "rating", "adult"]], y_pred], axis = 1)
netflix_pred_acc

In [ ]:
# Predict Probabilities corresponding to Predictors
y_prob = nbmodel.predict_proba(X_pred)
np.set_printoptions(precision = 3)
print(nbmodel.classes_)
print(y_prob)